# Resume ↔ Job Matching — Data Exploration

This notebook explores the two raw datasets before any cleaning or modeling:
- `postings.csv` — LinkedIn job postings (492MB, ~26,000+ rows)
- `Resume.csv` — labeled resume dataset

Goal: understand the columns, check data quality, and produce a filtered, manageable dataset for the rest of the project.

## 1. Inspect the postings file structure

The file is too large to load in full while exploring, so `nrows=0` reads only the header row — this tells us what columns exist without loading any actual data.

In [1]:
import pandas as pd

cols = pd.read_csv(r"D:\JOB\postings.csv", nrows=0).columns.tolist()
print(cols)

['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


## 2. Scan the full file for data-related roles

Rather than loading the whole 492MB file into memory, `chunksize=5000` reads it in batches of 5,000 rows. Each batch is filtered for job titles matching our target keywords, and only the matches are kept. This searches the **entire** file while only ever holding a small piece of it in memory at once.

We also drop `skills_desc` up front — an earlier check showed it's missing in 99.3% of rows, so it's not useful. Skills will instead be extracted from the `description` text in a later step.

In [2]:
keywords = "data scientist|data analyst|machine learning|data engineer"

chunks = []
for chunk in pd.read_csv(
    r"D:\JOB\postings.csv",
    usecols=["job_id", "title", "company_name", "location", "description", "formatted_experience_level"],
    chunksize=5000
):
    matched = chunk[chunk["title"].str.contains(keywords, case=False, na=False)]
    chunks.append(matched)

ds_postings = pd.concat(chunks, ignore_index=True)
print(ds_postings.shape)
ds_postings[["title", "company_name", "location"]].head(10)

(1187, 6)


,title,company_name,location
0,Sr Data Engineer with Kafka,ZenithMinds Inc,"Austin, TX"
1,Cloud Platform/ Big Data Engineer,"Subaru Research and Development, Inc","Michigan, United States"
2,Data Engineer/ETL,SilverSpace Technologies Inc,"Hartford, CT"
3,Data Analyst,Tenazx Inc,"Queens, NY"
4,Senior Data Engineer/Analyst - Full Time,NaN,"California, United States"
5,Senior Machine Learning Research Engineer,Symbolica AI,San Francisco Bay Area
6,Machine Learning Engineer,NLB Services,"Dallas, TX"
7,eCommerce Data Analyst,Radiant Systems Inc,United States
8,Azure Data Engineer (Full time),Publicis Sapient,"Arlington, VA"
9,Azure Data Engineer,"Econtenti, Inc","Seattle, WA"


## 3. Check data quality on the filtered set

Before saving, confirm there's nothing unexpected — missing descriptions, empty titles, or oddly short text that would break later steps.

In [3]:
print(ds_postings.isna().sum())
print()
print(ds_postings["description"].str.len().describe())

job_id                          0
company_name                   12
title                           0
description                     0
location                        0
formatted_experience_level    345
dtype: int64

count     1187.000000
mean      3624.684920
std       2340.669685
min         41.000000
25%       1645.500000
50%       3221.000000
75%       5214.500000
max      14748.000000
Name: description, dtype: float64


## 4. Save the filtered dataset

This is now a small, focused CSV (~1,200 rows instead of ~26,000+). From here on, load this file directly instead of re-scanning the original 492MB file.

In [4]:
ds_postings.to_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_postings_filtered.csv", index=False)

## 5. Load and inspect the resume dataset

`Resume.csv` is much smaller, so it can be loaded directly. Check its columns, size, and category distribution before cleaning.

In [5]:
resumes = pd.read_csv(r"D:\JOB\Resume\Resume.csv")
print(resumes.shape)
print(resumes.columns.tolist())
resumes.head()

(2484, 4)
['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
# Check for missing values and see how resumes are distributed across categories
print(resumes.isna().sum())
print()
print(resumes["Category"].value_counts())

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


## 6. Read a few full samples side by side

Before writing cleaning code, look at real examples. This surfaces boilerplate phrases, formatting artifacts, and how skills are typically mentioned — all of which inform what `cleaner.py` needs to strip out next.

In [7]:
print("--- SAMPLE JOB DESCRIPTION ---")
print(ds_postings["description"].iloc[0][:1000])

print("\n--- SAMPLE RESUME ---")
print(resumes["Resume_str"].iloc[0][:1000])

--- SAMPLE JOB DESCRIPTION ---
Data Engineer with Kafka (W2 Only)💯% Remote
Min 10 to12+ strong development experience neededVery strong experience in Kafka and Kafka data injection Strong exp in working with API.Strong exp in Python with AWS.Experience with Informatica IICS and Snowflake. Expertise in Snowflake's cloud data platform, including data loading, transformation, and querying using Snowflake SQL.Experience with SQL-based development, optimization, and tuning for large-scale data processing.Strong understanding of dimensional modeling concepts and experience in designing and implementing data models for analytics and reporting purposes.hands-on experience in IICS or Informatica Power Center ETL development1+ years of hands-on experience in Linux and shell scripting.1+ years of experience working with git.1+ years of related industry experience in an enterprise environment.1+ years of hands-on experience in Python programming.


--- SAMPLE RESUME ---
         HR ADMINISTRATOR/M